# Pipeline V8 — Fase 3: Re-rotulação Databricks (p=20, 11.542 estados)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-007). **Execução: Databricks.**

Re-calcula `melhor_jogada` e `score_melhor_jogada` somente para estados onde o
rótulo Minimax p=11 é potencialmente subótimo:

```
critério: arestas_livres > 11  E  prof_min > 11
onde:     arestas_livres = 31 - qtd_tracos
          prof_min       = total_caixas_cadeias_longas + 2 × qtd_cadeias_longas
```

**Profundidade única**: `p=20`. O Minimax para naturalmente quando o jogo termina;
p=20 resolve qualquer estado com ≤ 19 arestas livres (máximo observado no dataset).

**Estados a re-rotular** (análise 100% do dataset — 2026-05-14):

| arestas_livres | Estados |
|---|---|
| 12 | 3.601 |
| 13 | 3.000 |
| 14 | 2.232 |
| 15 | 1.581 |
| 16 | 836 |
| 17 | 244 |
| 18 | 45 |
| 19 | 3 |
| **Total** | **11.542** (1,52%) |

**Pré-requisito**: Fase 2 concluída (NPZs com campos `qtd_cadeias_longas` e
`total_caixas_cadeias_longas` presentes — schema v2-a3).

**Sobrescrita atômica**: `.tmp.npz` + `os.replace()`.

In [ ]:
# Configuração — ajustar paths conforme montagem no Databricks.

# Path do diretório NPZ no DBFS ou montagem S3.
DIR_NPZ = '/Workspace/Users/diondu@gmail.com/CNN/profundidade_minimax_11_v7_adaptativo'

# Profundidade única de re-rotulação.
DEPTH_REROTULACAO = 20

# Numero de workers Spark para mapInPandas.
N_PARTITIONS = 16

print(f'DIR_NPZ            = {DIR_NPZ}')
print(f'DEPTH_REROTULACAO  = {DEPTH_REROTULACAO}')
print(f'N_PARTITIONS       = {N_PARTITIONS}')

In [ ]:
# Imports
import os
import sys
import glob
import time
import numpy as np
import pandas as pd

# Em Databricks: o pacote deve estar instalado ou em sys.path via %pip install.
# Ajustar ROOT conforme o cluster.
ROOT = os.environ.get('ARENA_SAGAZ_BACKEND', '/home/ubuntu/arena-sagaz-backend')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from gerador_dados.jogo_pontinhos.minimax_pontinhos import calcular_jogada_minimax
from gerador_dados.jogo_pontinhos.tabuleiro_pontinhos import (
    labels_canonicos as LABELS_CANONICOS,
)

print('Imports OK.')

In [ ]:
# Diagnóstico: contar estados a re-rotular por arestas_livres.
# Gate T-A3-007: confirmar distribuição esperada antes de rodar.

SUFIXOS_SIMETRIA = ('_refH', '_refV', '_r180')
arquivos = sorted([
    f for f in glob.glob(os.path.join(DIR_NPZ, 'dataset_pequeno_*.npz'))
    if not any(os.path.splitext(os.path.basename(f))[0].endswith(s) for s in SUFIXOS_SIMETRIA)
])
print(f'NPZs originais: {len(arquivos)}')

from collections import Counter
contagem_por_arestas = Counter()

for arq in arquivos:
    with np.load(arq, allow_pickle=False) as d:
        qtd_tracos = d['qtd_tracos']              # (N,)
        qtd_cad    = d['qtd_cadeias_longas']       # (N,)
        total_cad  = d['total_caixas_cadeias_longas']  # (N,)

    arestas_livres = 31 - qtd_tracos.astype(np.int32)
    prof_min = total_cad.astype(np.int32) + 2 * qtd_cad.astype(np.int32)
    mascara = (arestas_livres > 11) & (prof_min > 11)

    for al in arestas_livres[mascara]:
        contagem_por_arestas[int(al)] += 1

total = sum(contagem_por_arestas.values())
print(f'\nEstados a re-rotular: {total:,} de {len(arquivos)*5000:,} ({100*total/(len(arquivos)*5000):.2f}%)')
print(f'\n{"arestas_livres":>16} | {"estados":>8}')
print('-' * 30)
for al in sorted(contagem_por_arestas):
    print(f'{al:>16} | {contagem_por_arestas[al]:>8}')
print(f'{"TOTAL":>16} | {total:>8}')

In [ ]:
# Função de re-rotulação de um único NPZ.

def rerotular_arquivo(caminho: str, depth: int) -> dict:
    """Re-calcula melhor_jogada e score_melhor_jogada para estados elegíveis.

    Critério: arestas_livres > 11 E prof_min > 11.
    Profundidade única p=depth (=20).
    Sobrescrita atômica.
    """
    t0 = time.time()
    with np.load(caminho, allow_pickle=False) as f:
        d = dict(f)

    estados    = d['estados']          # (N, 9, 7)
    qtd_tracos = d['qtd_tracos']       # (N,)
    qtd_cad    = d['qtd_cadeias_longas']
    total_cad  = d['total_caixas_cadeias_longas']
    melhor     = list(d['melhor_jogada'])
    scores     = d['score_melhor_jogada'].copy()  # (N, 31)

    arestas_livres = (31 - qtd_tracos.astype(np.int32))
    prof_min = total_cad.astype(np.int32) + 2 * qtd_cad.astype(np.int32)
    indices_elegíveis = np.where((arestas_livres > 11) & (prof_min > 11))[0]

    n_rerotulados = 0
    for i in indices_elegíveis:
        novos_scores, nova_jogada = calcular_jogada_minimax(estados[i], depth=depth)
        scores[i] = novos_scores
        melhor[i] = nova_jogada
        n_rerotulados += 1

    d['score_melhor_jogada'] = scores
    d['melhor_jogada']       = np.array(melhor)
    d['depth_melhor_jogada'] = np.where(
        (arestas_livres > 11) & (prof_min > 11),
        np.int8(depth),
        d['depth_melhor_jogada'],
    ).astype(np.int8)

    tmp = caminho + '.tmp.npz'
    np.savez_compressed(tmp, **d)
    os.replace(tmp, caminho)

    return {
        'caminho': caminho,
        'n_estados': len(estados),
        'n_rerotulados': n_rerotulados,
        'tempo_s': time.time() - t0,
    }

print('Funcao rerotular_arquivo() pronta.')

In [ ]:
# Loop principal de re-rotulação.
# Estimativa: ~11.542 chamadas Minimax p=20, ~30-60 min total em Databricks.

relatorio = []
for arq in arquivos:
    info = rerotular_arquivo(arq, depth=DEPTH_REROTULACAO)
    if info['n_rerotulados'] > 0:
        print(
            f"  {os.path.basename(info['caminho'])}: "
            f"re-rotulados={info['n_rerotulados']} "
            f"t={info['tempo_s']:.2f}s"
        )
    relatorio.append(info)

total_rerotulados = sum(r['n_rerotulados'] for r in relatorio)
tempo_total = sum(r['tempo_s'] for r in relatorio)

print()
print(f'NPZs processados   : {len(relatorio)}')
print(f'Estados re-rotulados: {total_rerotulados:,}')
print(f'Tempo total        : {tempo_total:.1f}s')

In [ ]:
# Auditoria pós-re-rotulação: verificar que nenhum .tmp residual existe.

tmps = [f + '.tmp.npz' for f in arquivos if os.path.exists(f + '.tmp.npz')]
if tmps:
    print(f'FALHA: {len(tmps)} arquivo(s) .tmp residual(is): {tmps[:5]}')
else:
    print(f'OK: {total_rerotulados:,} estados re-rotulados com p={DEPTH_REROTULACAO}.')
    print('Nenhum .tmp residual. Pronto para fase4_augmentacao_simetria.ipynb.')